# 02 - Exploratory Analysis
This notebook builds trend tables and figures for business insights.

In [ ]:

from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path(".").resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
FIGURES_DIR = PROJECT_ROOT / "deliverables" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEANED_DIR / "cta_ridership_cleaned.csv", parse_dates=["service_date"])


In [ ]:

annual = (
    df.groupby("year", as_index=False)
    .agg(
        total_annual_rides=("total_rides", "sum"),
        average_daily_rides=("total_rides", "mean"),
        total_bus_rides=("bus", "sum"),
        total_rail_rides=("rail_boardings", "sum"),
    )
    .sort_values("year")
)
annual["bus_share"] = annual["total_bus_rides"] / annual["total_annual_rides"]
annual["rail_share"] = annual["total_rail_rides"] / annual["total_annual_rides"]

monthly = (
    df.groupby(["year", "month", "month_name"], as_index=False)
    .agg(
        monthly_total_rides=("total_rides", "sum"),
        monthly_bus_rides=("bus", "sum"),
        monthly_rail_rides=("rail_boardings", "sum"),
    )
    .sort_values(["year", "month"])
)

baseline_2019 = annual.loc[annual["year"] == 2019, "total_annual_rides"].iloc[0]
recovery = annual.loc[annual["year"] >= 2020, ["year", "total_annual_rides"]].copy()
recovery["pct_of_2019"] = recovery["total_annual_rides"] / baseline_2019 * 100
recovery["pct_change_from_2019"] = (recovery["total_annual_rides"] - baseline_2019) / baseline_2019 * 100

day_comp = (
    df[df["year"].isin([2019, 2025])]
    .groupby(["year", "day_type_label"], as_index=False)
    .agg(avg_rides=("total_rides", "mean"))
)


In [ ]:

# Export summary tables
annual.to_csv(CLEANED_DIR / "annual_summary.csv", index=False)
monthly.to_csv(CLEANED_DIR / "monthly_summary.csv", index=False)
recovery.to_csv(CLEANED_DIR / "pandemic_recovery_summary.csv", index=False)
day_comp.to_csv(CLEANED_DIR / "day_type_comparison_2019_vs_2025.csv", index=False)


In [ ]:

# Matplotlib-only visuals
plt.style.use("ggplot")

plt.figure(figsize=(10, 5))
plt.plot(annual["year"], annual["total_annual_rides"] / 1_000_000, marker="o")
plt.title("CTA Annual Total Rides (Millions)")
plt.xlabel("Year"); plt.ylabel("Total Rides (Millions)")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "annual_total_rides_trend.png", dpi=150); plt.show()

plt.figure(figsize=(10, 5))
plt.plot(annual["year"], annual["total_bus_rides"] / 1_000_000, marker="o", label="Bus")
plt.plot(annual["year"], annual["total_rail_rides"] / 1_000_000, marker="o", label="Rail")
plt.title("CTA Annual Bus vs Rail Rides (Millions)")
plt.xlabel("Year"); plt.ylabel("Rides (Millions)"); plt.legend()
plt.tight_layout(); plt.savefig(FIGURES_DIR / "annual_bus_vs_rail_trend.png", dpi=150); plt.show()

pivot = day_comp.pivot(index="day_type_label", columns="year", values="avg_rides")
pivot.plot(kind="bar", figsize=(10,5), rot=0, title="Average Daily Rides by Day Type: 2019 vs 2025")
plt.xlabel("Day Type"); plt.ylabel("Average Rides")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "day_type_2019_vs_2025.png", dpi=150); plt.show()

month_order = ["January","February","March","April","May","June","July","August","September","October","November","December"]
seasonal = monthly.groupby("month_name", as_index=False)["monthly_total_rides"].mean()
seasonal["month_name"] = pd.Categorical(seasonal["month_name"], categories=month_order, ordered=True)
seasonal = seasonal.sort_values("month_name")
plt.figure(figsize=(11, 5))
plt.plot(seasonal["month_name"], seasonal["monthly_total_rides"] / 1_000_000, marker="o")
plt.title("CTA Monthly Ridership Seasonality (Average Monthly Rides)")
plt.xlabel("Month"); plt.ylabel("Average Monthly Rides (Millions)")
plt.xticks(rotation=45)
plt.tight_layout(); plt.savefig(FIGURES_DIR / "monthly_seasonality.png", dpi=150); plt.show()

plt.figure(figsize=(10, 5))
plt.bar(recovery["year"].astype(str), recovery["pct_of_2019"])
plt.axhline(100, color="black", linestyle="--", linewidth=1)
plt.title("CTA Recovery vs 2019 Baseline")
plt.xlabel("Year"); plt.ylabel("Ridership as % of 2019")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "recovery_pct_by_year.png", dpi=150); plt.show()
